# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [6]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

# Filter out documents with poor content quality and structure them properly
filtered_docs = []
for doc in loan_complaint_data:
    narrative = doc.metadata.get("Consumer complaint narrative", "")
    
    # Skip documents with insufficient content or too many redactions
    if (len(narrative.strip()) < 100 or 
        narrative.count("XXXX") > 5 or 
        narrative.strip() in ["", "None", "N/A"]):
        continue
    
    # Create meaningful page_content by combining narrative with context
    doc.page_content = f"Customer Issue: {doc.metadata.get('Issue', 'Unknown')}\n"
    doc.page_content += f"Product: {doc.metadata.get('Product', 'Unknown')}\n"
    doc.page_content += f"Complaint Details: {narrative}"
    
    filtered_docs.append(doc)

# Use filtered documents instead
loan_complaint_data = filtered_docs[:20]  # Start with smaller subset


Let's look at an example document to see if everything worked as expected!

In [7]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [9]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [10]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [13]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [15]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [16]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to "Dealing with your lender or servicer." Specific sub-issues frequently mentioned include trouble with how payments are being handled, receiving bad or conflicting information about the loan, issues with loan balances or terms, and problems with incorrect or outdated information on credit reports. Many complaints also involve lack of transparency, difficulty in communication, and concerns over unauthorized access or data breaches.\n\nIn summary, a prevalent issue is difficulty and frustration arising from interactions with loan lenders or servicers, including payment processing problems, misinformation, and poor communication.'

In [17]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, all responses from the companies indicate that they responded in a timely manner ("Company response to consumer": "Closed with explanation" and "Timely response?": "Yes"). Therefore, there do not appear to be any complaints that were not handled in a timely manner.'

In [18]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with how payments are being handled or re-amortized after forbearance periods, receiving incorrect or unclear information about their loan status, problems with automatic payments not processing correctly, and complications resulting from transfers or mismanagement by loan servicers. Some also faced difficulties due to unauthorized access to their personal information and lack of transparency from lenders or servicers, which contributed to misunderstandings about their loan obligations.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [19]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [20]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [21]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issue with loans appears to be difficulties in dealing with lenders or servicers, particularly related to how payments are being handled. Many complaints mention problems such as payments not being properly applied, issues with auto-pay reversals, confusion about payment statuses, and a lack of transparency or communication from the lenders or servicers.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all the complaints listed show responses from the companies that were marked as "Timely response" (all indicated as "Yes"). Therefore, there is no evidence in this data that any complaints were not handled in a timely manner.'

In [23]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues related to the management and handling of their payments by servicers, as well as problems with the accuracy and reporting of their account statuses. For example, one individual set up autopay for their student loans but was told it was incomplete and did not process, resulting in a past due notice despite the autopay being in place. Others experienced reversals or failures in automatic payments, leading to higher owed amounts or being reported as delinquent or in default. Additionally, there were cases where borrowers found out about their default status only through credit reports, often without prior notice, which adversely affected their credit scores and financial opportunities. Overall, these issues stem from administrative errors, insufficient notifications, or mishandling of payment processing by loan servicers.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
####✅ Answer: What was the weather in Charlotte yesterday? Because BM25 works better when the query need retrival based on precise keyword matches  vs abstract and semantic generalization needs

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [24]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [25]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided context, appears to be difficulties in dealing with lenders or servicers, particularly related to payment handling problems. This includes issues such as payments not being processed correctly, auto-pay reversals, and challenges in managing or understanding repayment processes.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that some complaints were handled in a timely manner, as indicated by the responses marked as "Yes" for whether the response was timely. However, the third complaint about autopay issues was also responded to quickly and explained, suggesting it was handled promptly.\n\nThe first complaint regarding the customer service representative disconnecting the call without resolving the issue was marked "Yes" for a timely response, but there is no explicit information about whether the issue itself was resolved promptly. The second complaint about data breaches was also marked "Yes" for response timeliness, but it primarily pertains to reporting violations and requesting investigations.\n\nGiven the information, there is no clear evidence of any complaints not being handled in a timely manner. All complaints referenced appear to have received responses within the expected timeframe.\n\n**Therefore, based on the data provided, I cannot determine tha

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including issues with the handling of payments by lenders or servicers, technical or administrative problems, and misunderstandings about account status. For example, some borrowers experienced autopay setups not being processed correctly, leading to missed payments and overdue notices. Others found their loans were incorrectly reported as in default or delinquent, despite never having missed payments, which negatively affected their credit scores and financial opportunities. Overall, problems with loan administration, lack of timely notifications, and errors in reporting contributed to borrowers' inability to repay their loans successfully."

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [29]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [30]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [31]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided complaints, a most common issue with loans appears to be problems related to dealing with lenders or servicers, particularly issues such as:\n\n- Receiving bad or incorrect information about the loans\n- Trouble with how payments are being handled, including autopay setup failures, payment reversals, or misapplied payments\n- Lack of communication or transparency, including not receiving notices about loan transfers or payment status\n- Discrepancies in loan balance, interest, or account status\n- Issues with loan account status reporting, such as incorrect defaults or delinquency notifications\n- Data breaches and unauthorized access compromising borrower information\n\nMany complaints indicate frustration with poor communication, administrative errors, or mismanagement by loan servicers, which significantly impacts borrowers' understanding and ability to manage their loans effectively.\n\nIf you need a more specific summary, please let me know!"

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, all of them were responded to in a timely manner. Each complaint received a response from the companies within the appropriate timeframe, and the responses cited indicate that responses were completed and closed with explanation, confirming that no complaints remained unhandled in a timely manner.'

In [33]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons. Based on the provided complaints, some common reasons include:\n\n1. **Incorrect or Mismanaged Information:** Borrowers found out their loans were in default or delinquent due to incorrect reporting or processing errors, such as miscommunication about account status or incorrect reporting of past due accounts, which impacted their credit scores and eligibility.\n\n2. **Lack of Transparency or Poor Communication:** Borrowers experienced difficulty obtaining clear information about their loan status, servicer changes, or repayment terms, leading to confusion and missed payments.\n\n3. **Problems with Loan Servicers and Payment Handling:** Issues like failed automatic payments despite setup, payment reversals, or incorrect billing amounts caused borrowers to fall behind.\n\n4. **Legal or Data Privacy Violations:** Some borrowers faced issues due to data breaches, unauthorized access, or mishandling of personal information, which 

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
###✅ Answer: I can't expect the user to be trained on prompting, and my app should handle queries written in all type of prose and loaded with errors. query reformulation would allow the retriver to retrive relevant information that semantically matches several reforumulations of the original query (ideally matching the actul intent of the query better). It increases retrival diversity, by retriving context that may come from lexically variant sources.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [34]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [35]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [36]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [37]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [38]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [39]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, a common issue with loans—particularly student loans in this context—is problems related to dealing with lenders or servicers. Specific issues include trouble with how payments are being handled (such as autopay failures, payment reversals, or incorrect billing) and communication problems like lack of transparency, poor customer service, or difficulty obtaining accurate information. These issues can cause significant stress and financial complications for borrowers.'

In [40]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints and their statuses, all of the complaints shown in the context indicate that the responses from the companies were marked as "Closed with explanation" and responded to in a timely manner ("Yes" under "Timely response?"). There is no indication that any complaints were left unhandled or not addressed in a timely manner.'

In [41]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"Based on the provided context, people failed to pay back their loans for multiple reasons, including:\n\n1. **Disputes over the legitimacy and legality of the debt**: Some borrowers believe that certain student loans are now legally void due to the abolishment of the Department of Education, yet are still being reported and collected illegally.\n\n2. **Poor communication and transparency from loan servicers**: Borrowers report difficulty in obtaining clear information about their loan status, repayment terms, or whether they are in forbearance, leading to confusion and possibly missed or delayed payments.\n\n3. **Issues related to data breaches and mishandling of personal information**: Concerns over data security and privacy breaches may undermine trust in the loan management process, potentially affecting repayment.\n\n4. **Financial difficulties and increased payment amounts**: Changes in repayment conditions, such as re-amortization after forbearance, can cause significant increas

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [42]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [44]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [45]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided context, appears to be problems related to "Dealing with your lender or servicer." Specific sub-issues frequently mentioned include trouble with how payments are being handled, receiving bad information about loans, problems with repayment plans and billing, and issues with data security and privacy breaches. Many complaints involve mismanagement of payments, incorrect account information, and mishandling of personal data.'

In [46]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided in the complaints, several complaints indicate issues with timeliness or handling of their cases. Specifically:\n\n- The complaint received on 04/28/25 about autopay setup showed that the consumer did not receive notifications that autopay was incomplete, but the response was marked as "Timely" and "Closed with explanation."\n- Other complaints, such as the one received on 04/14/25 regarding violations and breaches, also received responses marked as "Timely" and "Closed with explanation."\n- Multiple complaints mention unresolved issues, mismanagement, or delays, but the responses from the companies were still marked as timely.\n\nHowever, some complaints explicitly describe situations where consumers experienced failures or delays in handling their issues, such as failed communication, unexplained discharges, or billing errors. While these highlight problems with handling, the official response status suggests that, at least from the company\'s persp

In [47]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including:\n\n- **Incorrect or mismanaged information:** Some borrowers found out their accounts were in default due to errors or misreporting by loan servicers, despite never being in default themselves.\n- **Problems with payment handling:** Issues like autopay setup failures, processing delays, or technical errors prevented automatic payments from being deducted.\n- **Lack of communication or transparency:** Borrowers experienced long wait times, lack of notifications, or inconsistent information from servicers about their loan status or repayment obligations.\n- **Data breaches and security issues:** Unauthorized access and mishandling of personal information compromised borrower data, leading to confusion and difficulties in managing loans.\n- **Legal or administrative complications:** Changes such as loan transfers, disputes over payment records, or the ending of programs (like COVID-19 forbearance) without proper re-amo

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [48]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [49]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [50]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [51]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [52]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [53]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans, particularly student loans in this dataset, seem to involve dealing with lenders or servicers, problems with payment handling, incorrect information on credit reports, data breaches, and difficulties in repayment, including issues with loan forgiveness or discharge. Many complaints also involve lack of transparency, unauthorized data access, and mismanagement of borrower information.\n\nIn summary, the most common issues are related to:\n\n- Difficulties in managing and handling payments (e.g., reversals, incorrect amounts)\n- Problems with loan status information on credit reports (e.g., default, delinquency)\n- Challenges in communication and transparency from servicers\n- Data breaches and unauthorized access to personal information\n- Disputes over loan amounts and payment records \n\nHowever, if we consider the overarching theme, the most frequently mentioned problem appears to be **"Dealing with your lender or ser

In [54]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were responded to in a timely manner, as indicated by the "Timely response?" field being marked "Yes" for each case. The responses given to these complaints were typically "Closed with explanation" or "None," but importantly, they received responses within the expected timeframe.\n\nTherefore, there is no evidence in this data suggesting that any complaints did not get handled in a timely manner.'

In [55]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with how payments were being handled or re-amortized after forbearance ended, receiving incorrect or bad information about their loan status, technical problems with payment processing (such as reversals or failure to deduct payments), data breaches or mishandling of personal information, and disputes over the legitimacy of their debt or recent changes in loan status. Additionally, some borrowers faced difficulties due to lack of transparency, inadequate communication from lenders or servicers, or errors in their account information that affected their credit reports and ability to repay.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
### ✅✅✅ Answer: If questions are repetitive, semantic chunking will merge the questions into similar chunks, losing distinction of questions, and providing redundant, blended, generic answers. One way is to not use semantic chunking for questions in FAQ, but alternatively, reduce the chunk size with clear seperators to presever the units within each FAQ. Second option is to treat each questions as a separate chunk. 

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [56]:
#NLTK Import To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/chrag/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/chrag/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [57]:
### Step 0: Dependencies & imports

# included in pyproject.toml file
# "numpy>=2.2.2",
# "ragas==0.3.0",
# "rapidfuzz"
# "langchain-core",
# "langchain-community",
# "langsmith",
#  "tqdm", 


#Setting up the LLM and embedding model and generator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


# Use a more capable model for complex knowledge extraction tasks
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",  # More capable than nano for reasoning tasks
    temperature=0.1,      # Lower temperature for more consistent outputs
    request_timeout=120   # Longer timeout for complex operations
))

generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [58]:
#Step 1 Create a "golden dataset" a.k.a synthetic test data
# This will generate our knowledge graph under the hood and generate our personas and scenarios to construct our queries

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
sample_docs = loan_complaint_data[:15]  # subset of full data set

from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]


try:
    print(f"Generating test dataset with {len(sample_docs)} documents...")
    dataset = generator.generate_with_langchain_docs(
        sample_docs,
        testset_size=10,
        query_distribution=query_distribution,  # adding query distribution to the generator
    )
    print("Dataset generation completed successfully!")
    print(f"Generated {len(dataset)} test samples")
except Exception as e:
    print(f"Error during dataset generation: {e}")
    print("Try reducing document count or testset_size further")

Generating test dataset with 15 documents...


Applying SummaryExtractor:   0%|          | 0/7 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/15 [00:00<?, ?it/s]

Node a7d41a5f-2758-492e-894f-b359f131d219 does not have a summary. Skipping filtering.
Node dc88982e-3109-4e30-8c58-f3eade5daa82 does not have a summary. Skipping filtering.
Node ec9b92da-b66d-4e98-8a9d-f5c5691eaadc does not have a summary. Skipping filtering.
Node d26052de-82cc-485a-b913-67ee3cebd4eb does not have a summary. Skipping filtering.
Node 1ed6e62f-5d46-4e2e-9a44-e3a1b1652009 does not have a summary. Skipping filtering.
Node 4f4a1526-c06b-47bd-ae5f-cc3cfebfc4f0 does not have a summary. Skipping filtering.
Node f79b5a3a-8329-4f5c-bda8-02ab9f76d179 does not have a summary. Skipping filtering.
Node 120555cf-4a54-49e9-8766-da22f0d1eaf7 does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/37 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

Dataset generation completed successfully!
Generated 11 test samples


In [60]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What issues are borrowers facing with Nelnet a...,[Customer Issue: Dealing with your lender or s...,Borrowers are facing significant issues with N...,single_hop_specifc_query_synthesizer
1,What is the problem with the Income-Driven Rep...,[Customer Issue: Dealing with your lender or s...,The problem with the Income-Driven Repayment p...,single_hop_specifc_query_synthesizer
2,What complaint was made regarding the student ...,[Customer Issue: Dealing with your lender or s...,The complaint details that the borrower's pers...,single_hop_specifc_query_synthesizer
3,What does Studentaid.gov say about changes to ...,[Customer Issue: Dealing with your lender or s...,"According to Studentaid.gov, you should receiv...",single_hop_specifc_query_synthesizer
4,Wut does forbearance until 2040 mean for my st...,[Customer Issue: Dealing with your lender or s...,Forbearance until 2040 means that your federal...,single_hop_specifc_query_synthesizer
5,What issues are customers facing with student ...,[<1-hop>\n\nCustomer Issue: Improper use of yo...,Customers are facing significant issues with s...,multi_hop_abstract_query_synthesizer
6,What are the common customer complaints regard...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Common customer complaints regarding Aidvantag...,multi_hop_abstract_query_synthesizer
7,How can I dispute an incorrect student loan pa...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,To dispute the incorrect student loan payment ...,multi_hop_abstract_query_synthesizer
8,What issues did customers face with AidVantage...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Customers faced significant issues with AidVan...,multi_hop_specific_query_synthesizer
9,What issues are borrowers facing with Nelnet r...,[<1-hop>\n\nCustomer Issue: Dealing with your ...,Borrowers are facing significant issues with N...,multi_hop_specific_query_synthesizer


In [86]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your Langchain API Key:")
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = f"AIM - StudentLoanRetrievalEval - {uuid4().hex[0:8]}"
from langchain.callbacks.tracers import LangChainTracer
from langchain.schema.runnable import RunnableConfig

In [91]:
import copy
import time
from ragas.metrics import LLMContextRecall,ContextEntityRecall,LLMContextPrecisionWithReference,NonLLMContextPrecisionWithReference
from ragas import RunConfig
from uuid import uuid4



custom_run_config = RunConfig(timeout=360)


retrievers = {
    "Naive": naive_retriever,
    "BM25": bm25_retriever,
    "ContextualCompression": compression_retriever,
    "MultiQuery": multi_query_retriever,
    "ParentDocument": parent_document_retriever,
    "Ensemble": ensemble_retriever
}

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

retriever_results = {}

#tracer = LangChainTracer(project_name=f"{os.environ["LANGCHAIN_PROJECT"]} - {retriever_name}")


for retriever_name, retriever in retrievers.items():
    print(f"Evaluating {retriever_name}...")

#Lets make a deep copy of test dataset
    test_dataset_copy = copy.deepcopy(dataset)

    for test_row in test_dataset_copy:
        # Add rate limiting before calling any retriever that uses Cohere
        if retriever_name in ["ContextualCompression", "Ensemble"]:
            time.sleep(6.1)  # Wait 6.1 seconds to stay under 10 calls/minute
        docs = retriever.invoke(test_row.eval_sample.user_input)
        test_row.eval_sample.retrieved_contexts = [doc.page_content for doc in docs]
        #  contexts.append(context)
 
    evaluation_dataset = EvaluationDataset.from_pandas(test_dataset_copy.to_pandas())
    
    tracer = LangChainTracer(project_name=f"{os.environ["LANGCHAIN_PROJECT"]} - {retriever_name}")

    # Evaluate THIS retriever
    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(),ContextEntityRecall(),LLMContextPrecisionWithReference(),NonLLMContextPrecisionWithReference()],
        llm=evaluator_llm,
        run_config=custom_run_config,
        callbacks=[tracer]
    )
    
    retriever_results[retriever_name] = result

print(retriever_results)

Evaluating Naive...


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

Evaluating BM25...


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

Evaluating ContextualCompression...


...

Evaluating ParentDocument...


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

Evaluating Ensemble...


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

{'Naive': {'context_recall': 1.0000, 'context_entity_recall': 0.4440, 'llm_context_precision_with_reference': 0.6624, 'non_llm_context_precision_with_reference': 0.7485}, 'BM25': {'context_recall': 0.8333, 'context_entity_recall': 0.4655, 'llm_context_precision_with_reference': 0.8258, 'non_llm_context_precision_with_reference': 0.6818}, 'ContextualCompression': {'context_recall': 0.9697, 'context_entity_recall': 0.5038, 'llm_context_precision_with_reference': 0.8333, 'non_llm_context_precision_with_reference': 0.9242}, 'MultiQuery': {'context_recall': 1.0000, 'context_entity_recall': 0.4646, 'llm_context_precision_with_reference': 0.7459, 'non_llm_context_precision_with_reference': 0.7378}, 'ParentDocument': {'context_recall': 0.8545, 'context_entity_recall': 0.4419, 'llm_context_precision_with_reference': 0.7424, 'non_llm_context_precision_with_reference': 0.6818}, 'Ensemble': {'context_recall': 0.9545, 'context_entity_recall': 0.4536, 'llm_context_precision_with_reference': 0.7296, 

### ✅✅✅ ANALYSIS 

| Retriever              | Context Recall | Entity Recall | LLM Precision | Non-LLM Precision | Latency (s) | Output Tokens | Total Tokens | Cost ($) |
|------------------------|----------------|----------------|----------------|--------------------|--------------|----------------|----------------|-----------|
| ContextualCompression  | 0.9697         | **0.5038**     | **0.8333**     | **0.9242**         | **8.31**     | 13,149         | 165,283        | **0.07**   |
| BM25                   | 0.8333         | 0.4655         | **0.8258**     | 0.6818             | 10.84        | 24,997         | 355,516        | 0.14      |
| MultiQuery             | **1.0000**     | 0.4646         | 0.7459         | 0.7378             | 30.39        | 33,444         | 505,519        | 0.20      |
| ParentDocument         | 0.8545         | 0.4419         | 0.7424         | 0.6818             | 23.49        | 15,739         | 208,771        | 0.09      |
| Ensemble               | 0.9545         | 0.4536         | 0.7296         | 0.7778             | 31.90        | 18,115         | 292,613        | 0.11      |
| Naive                  | **1.0000**     | 0.4440         | 0.6624         | 0.7485             | 21.66        | **41,655**     | **600,492**    | **0.24**  |


🔚 Summary Recommendations
Retriever	Best For	Notes
ContextualCompression	🏆 Overall best	Fast, cheap, and top-tier precision
BM25	🧠 High LLM precision, low cost	Best simple baseline with speed
Ensemble	⚙️ Robust combo	Slower, moderate cost
MultiQuery	📚 Recall-focused	Overkill unless max recall is essential
Naive	❌ Avoid	Low precision, highest cost